# Phase 1 — Preprocessing Verification Notebook

Verifies all preprocessing pipeline outputs: feature shapes, normalization stats,
split integrity, and visualizes sample spectrograms/MFCCs.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import preprocess_audio, TARGET_SR, TARGET_SAMPLES
from features import compute_log_mel_spectrogram, compute_mfcc

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
MEL_DIR = PROJECT_ROOT / "data" / "features" / "mel_spectrograms"
MFCC_DIR = PROJECT_ROOT / "data" / "features" / "mfcc"
CLASSICAL_DIR = PROJECT_ROOT / "data" / "classical_ml_features"
DATA_DIR = PROJECT_ROOT / "data"

with open(DATA_DIR / "preprocessing_config.json") as f:
    config = json.load(f)

print("Preprocessing config loaded.")
print(f"  Target SR: {config['target_sr']} Hz")
print(f"  Target duration: {config['target_duration_s']}s ({config['target_samples']} samples)")
print(f"  Mel shape: {config['mel_spectrogram']['output_shape']}")
print(f"  MFCC shape: {config['mfcc']['output_shape']}")
print(f"  Classical features: {config['classical_ml_features']['total_features']}-dim")

## 1. Manifest Summary & Class Distributions

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / "train_manifest.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_manifest.csv")
test_df = pd.read_csv(SPLITS_DIR / "test_manifest.csv")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
assert len(train_df) == 970, f"Expected 970 train, got {len(train_df)}"
assert len(val_df) == 208, f"Expected 208 val, got {len(val_df)}"
assert len(test_df) == 208, f"Expected 208 test, got {len(test_df)}"
print("PASS: Split counts match expectations.")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (tier, tier_col, n_classes) in zip(axes, [
    ("Tier 1 (Binary)", "tier1_name", 2),
    ("Tier 2 (6-class)", "tier2_name", 6),
    ("Tier 3 (9-class, excl. combined)", "tier3_name", 9),
]):
    for split_name, df, color in [("Train", train_df, "#4C72B0"),
                                   ("Val", val_df, "#55A868"),
                                   ("Test", test_df, "#C44E52")]:
        if tier == "Tier 3 (9-class, excl. combined)":
            df = df[df["tier3_label"] >= 0]
        counts = df[tier_col].value_counts().sort_index()
        ax.barh(counts.index, counts.values, alpha=0.7, label=split_name)
    ax.set_title(tier)
    ax.set_xlabel("Count")
    ax.legend()

plt.tight_layout()
plt.show()

## 2. Audio Preprocessing Verification

Pick one file from each operational state (braking, idle, startup) and verify:
- Output shape is (24000,)
- Peak amplitude is 1.0
- Startup clips were center-cropped from ~1.7s

In [ ]:
sample_files = {
    "braking": train_df[train_df["state"] == "braking state"].iloc[0],
    "idle": train_df[train_df["state"] == "idle state"].iloc[0],
    "startup": train_df[train_df["state"] == "startup state"].iloc[0],
}

fig, axes = plt.subplots(3, 2, figsize=(14, 9))

for i, (state, row) in enumerate(sample_files.items()):
    full_path = str(PROJECT_ROOT / row["file_path"])

    # Raw audio
    y_raw, sr_raw = librosa.load(full_path, sr=None, mono=True)
    axes[i, 0].plot(np.arange(len(y_raw)) / sr_raw, y_raw, linewidth=0.5)
    axes[i, 0].set_title(f"Raw: {state} (sr={sr_raw}, dur={len(y_raw)/sr_raw:.3f}s)")
    axes[i, 0].set_ylabel("Amplitude")

    # Preprocessed audio
    y_proc = preprocess_audio(full_path)
    axes[i, 1].plot(np.arange(len(y_proc)) / TARGET_SR, y_proc, linewidth=0.5, color="C1")
    axes[i, 1].set_title(f"Preprocessed: shape={y_proc.shape}, peak={np.max(np.abs(y_proc)):.4f}")
    axes[i, 1].set_ylabel("Amplitude")

    # Assertions
    assert y_proc.shape == (TARGET_SAMPLES,), f"Shape mismatch: {y_proc.shape}"
    assert np.isclose(np.max(np.abs(y_proc)), 1.0, atol=1e-6), f"Peak not 1.0: {np.max(np.abs(y_proc))}"

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

print("PASS: All preprocessed clips have shape (24000,) and peak amplitude 1.0.")

## 3. Mel-Spectrogram Verification

Verify shapes and visualize one sample per Tier 2 class.

In [ ]:
mel_train = np.load(MEL_DIR / "train.npz")
X_mel = mel_train["X"]
y_tier2 = mel_train["y_tier2"]

print(f"Mel train shape: {X_mel.shape}, dtype: {X_mel.dtype}")
print(f"Value range: [{X_mel.min():.1f}, {X_mel.max():.1f}] dB")
assert X_mel.shape == (970, 40, 92), f"Unexpected shape: {X_mel.shape}"
print("PASS: Mel-spectrogram shape is (970, 40, 92).")

tier2_names = config["tier_definitions"]["tier2"]["names"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for label_idx, ax in enumerate(axes.flat):
    sample_idx = np.where(y_tier2 == label_idx)[0][0]
    img = librosa.display.specshow(
        X_mel[sample_idx], sr=TARGET_SR,
        hop_length=config["mel_spectrogram"]["hop_length"],
        x_axis="time", y_axis="mel", ax=ax,
        fmin=config["mel_spectrogram"]["fmin"],
        fmax=config["mel_spectrogram"]["fmax"],
    )
    ax.set_title(f"[{label_idx}] {tier2_names[label_idx]}")
    plt.colorbar(img, ax=ax, format="%+2.0f dB")

plt.suptitle("Log-Mel Spectrograms — One per Tier 2 Class", fontsize=14)
plt.tight_layout()
plt.show()

## 4. MFCC Verification

In [ ]:
mfcc_train = np.load(MFCC_DIR / "train.npz")
X_mfcc = mfcc_train["X"]

print(f"MFCC train shape: {X_mfcc.shape}, dtype: {X_mfcc.dtype}")
assert X_mfcc.shape == (970, 92, 13), f"Unexpected shape: {X_mfcc.shape}"
print("PASS: MFCC shape is (970, 92, 13).")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
y_tier2_mfcc = mfcc_train["y_tier2"]

for label_idx, ax in enumerate(axes.flat):
    sample_idx = np.where(y_tier2_mfcc == label_idx)[0][0]
    # Transpose back to (13, 92) for display
    img = ax.imshow(X_mfcc[sample_idx].T, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(f"[{label_idx}] {tier2_names[label_idx]}")
    ax.set_xlabel("Frame")
    ax.set_ylabel("MFCC Coefficient")
    plt.colorbar(img, ax=ax)

plt.suptitle("MFCCs — One per Tier 2 Class", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Classical ML Feature Verification

In [ ]:
classical_train = np.load(CLASSICAL_DIR / "train.npz")
X_classical = classical_train["X"]

print(f"Classical train shape: {X_classical.shape}, dtype: {X_classical.dtype}")
assert X_classical.shape == (970, 441), f"Unexpected shape: {X_classical.shape}"

n_nan = np.isnan(X_classical).sum()
n_inf = np.isinf(X_classical).sum()
print(f"NaN count: {n_nan} | Inf count: {n_inf}")
assert n_nan == 0, f"Found {n_nan} NaN values!"
assert n_inf == 0, f"Found {n_inf} Inf values!"
print("PASS: Classical features shape (970, 441), no NaN/Inf.")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(X_classical.flatten(), bins=100, edgecolor="none", alpha=0.7)
ax.set_xlabel("Feature Value")
ax.set_ylabel("Count")
ax.set_title("Distribution of All 441-dim Classical ML Features (Training Set)")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 6. Normalization Stats Verification

In [ ]:
norm_stats = np.load(DATA_DIR / "normalization_stats.npz")

mel_mean = norm_stats["mel_mean"]
mel_std = norm_stats["mel_std"]
mfcc_mean = norm_stats["mfcc_mean"]
mfcc_std = norm_stats["mfcc_std"]

print(f"Mel mean shape: {mel_mean.shape}, std shape: {mel_std.shape}")
print(f"MFCC mean shape: {mfcc_mean.shape}, std shape: {mfcc_std.shape}")
assert mel_mean.shape == (40,) and mel_std.shape == (40,)
assert mfcc_mean.shape == (13,) and mfcc_std.shape == (13,)
assert np.all(mel_std > 0), "Some mel std values are <= 0!"
assert np.all(mfcc_std > 0), "Some MFCC std values are <= 0!"
print("PASS: Normalization stats have correct shapes and all std > 0.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(mel_mean, label="Mean", marker="o", markersize=3)
axes[0].fill_between(range(40), mel_mean - mel_std, mel_mean + mel_std, alpha=0.3, label="Mean +/- Std")
axes[0].set_xlabel("Mel Band")
axes[0].set_ylabel("dB")
axes[0].set_title("Per-Mel-Band Normalization Stats (Training Set)")
axes[0].legend()

axes[1].bar(range(13), mfcc_mean, yerr=mfcc_std, capsize=3, alpha=0.7)
axes[1].set_xlabel("MFCC Coefficient")
axes[1].set_ylabel("Value")
axes[1].set_title("Per-MFCC Normalization Stats (Training Set)")
axes[1].set_xticks(range(13))

plt.tight_layout()
plt.show()

## 7. Tier 3 Exclusion Check

In [ ]:
for split_name, n_expected_total, n_expected_valid in [
    ("train", 970, 664), ("val", 208, 142), ("test", 208, 143)
]:
    data = np.load(MEL_DIR / f"{split_name}.npz")
    y_t3 = data["y_tier3"]
    n_total = len(y_t3)
    n_excluded = np.sum(y_t3 == -1)
    n_valid = np.sum(y_t3 >= 0)
    print(f"  {split_name}: total={n_total}, excluded={n_excluded}, valid_tier3={n_valid}")
    assert n_total == n_expected_total, f"Total mismatch for {split_name}"
    assert n_valid == n_expected_valid, f"Valid Tier 3 count mismatch for {split_name}: got {n_valid}"

print("PASS: Tier 3 exclusion counts match expectations (664/142/143).")

## 8. Cross-Split Leakage Check

Verify no file appears in more than one split.

In [ ]:
train_paths = set(train_df["file_path"])
val_paths = set(val_df["file_path"])
test_paths = set(test_df["file_path"])

train_val = train_paths & val_paths
train_test = train_paths & test_paths
val_test = val_paths & test_paths

print(f"Train-Val overlap: {len(train_val)}")
print(f"Train-Test overlap: {len(train_test)}")
print(f"Val-Test overlap: {len(val_test)}")
assert len(train_val) == 0, f"Leakage: {len(train_val)} files in both train and val!"
assert len(train_test) == 0, f"Leakage: {len(train_test)} files in both train and test!"
assert len(val_test) == 0, f"Leakage: {len(val_test)} files in both val and test!"

total_unique = len(train_paths | val_paths | test_paths)
print(f"Total unique files across all splits: {total_unique}")
assert total_unique == 1386, f"Expected 1386 total files, got {total_unique}"
print("PASS: No cross-split leakage. All 1386 files accounted for.")

## 9. Val/Test Feature Shape Consistency

In [ ]:
checks = [
    ("mel_spectrograms/val.npz", MEL_DIR / "val.npz", (208, 40, 92)),
    ("mel_spectrograms/test.npz", MEL_DIR / "test.npz", (208, 40, 92)),
    ("mfcc/val.npz", MFCC_DIR / "val.npz", (208, 92, 13)),
    ("mfcc/test.npz", MFCC_DIR / "test.npz", (208, 92, 13)),
    ("classical/val.npz", CLASSICAL_DIR / "val.npz", (208, 441)),
    ("classical/test.npz", CLASSICAL_DIR / "test.npz", (208, 441)),
]

all_pass = True
for name, path, expected_shape in checks:
    data = np.load(path)
    actual_shape = data["X"].shape
    status = "PASS" if actual_shape == expected_shape else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  {status}: {name} — shape {actual_shape} (expected {expected_shape})")

assert all_pass, "Some shape checks failed!"
print("\nPASS: All val/test feature shapes are correct.")

## 10. Summary

In [ ]:
summary = """
Phase 1 Preprocessing Verification — All Checks Passed
========================================================
 1. Split counts:          970 train / 208 val / 208 test
 2. Audio preprocessing:   shape (24000,), peak=1.0, center-cropped
 3. Mel-spectrograms:      (N, 40, 92) float32, range ~[-80, 0] dB
 4. MFCCs:                 (N, 92, 13) float32
 5. Classical ML features: (N, 441) float32, no NaN/Inf
 6. Normalization stats:   mel (40,), mfcc (13,), all std > 0
 7. Tier 3 exclusion:      664/142/143 valid samples
 8. Cross-split leakage:   None — all 1386 files in exactly one split
 9. Val/Test consistency:  All shapes match expectations
"""
print(summary)